In [1]:
!pip install pymupdf langchain langchain-community beautifulsoup4 requests nltk tiktoken -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [4]:
!pip install langchain langchain-text-splitters -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Vedant_Roadmap_FINAL.pdf to Vedant_Roadmap_FINAL.pdf


In [5]:
import fitz
import tiktoken
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader

# Token counter — use this to verify actual token counts
def count_tokens(text, model="gpt-3.5-turbo"):
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

# Load PDF
def load_pdf(path):
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    print(f"Loaded {len(docs)} pages")
    return docs

# Load webpage
def load_webpage(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    for tag in soup(['nav', 'footer', 'header', 'script', 'style']):
        tag.decompose()
    text = soup.get_text(separator='\n', strip=True)
    print(f"Loaded webpage: {len(text)} characters")
    return text

# Chunk documents
def chunk_documents(docs, chunk_size=512, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=count_tokens,  # measure in tokens, not characters
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_documents(docs)
    print(f"Created {len(chunks)} chunks")
    return chunks

# Chunk size experiment
def chunk_size_experiment(text, sizes=[128, 256, 512, 1024]):
    print(f"\nTotal text tokens: {count_tokens(text)}\n")
    print(f"{'Chunk Size':<15} {'Num Chunks':<15} {'Avg Tokens/Chunk':<20}")
    print("-" * 50)

    for size in sizes:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=size,
            chunk_overlap=int(size * 0.1),
            length_function=count_tokens
        )
        chunks = splitter.split_text(text)
        avg_tokens = sum(count_tokens(c) for c in chunks) / len(chunks)
        print(f"{size:<15} {len(chunks):<15} {avg_tokens:<20.1f}")

In [6]:
# Test on any PDF you have — your roadmap PDF works fine
docs = load_pdf("Vedant_Roadmap_FINAL.pdf")
full_text = " ".join([d.page_content for d in docs])

# See how chunk size affects number of chunks
chunk_size_experiment(full_text, sizes=[128, 256, 512, 1024])

# Produce final chunks for Week 4 Day 3 (retrieval pipeline)
chunks = chunk_documents(docs, chunk_size=512, chunk_overlap=50)

# Inspect a few chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Tokens: {count_tokens(chunk.page_content)}")
    print(f"Source: {chunk.metadata.get('source', 'unknown')}, Page: {chunk.metadata.get('page', '?')}")
    print(chunk.page_content[:200])

Loaded 10 pages

Total text tokens: 4611

Chunk Size      Num Chunks      Avg Tokens/Chunk    
--------------------------------------------------
128             41              117.2               
256             20              244.8               
512             11              459.7               
1024            5               996.6               
Created 15 chunks

--- Chunk 1 ---
Tokens: 288
Source: Vedant_Roadmap_FINAL.pdf, Page: 0
Vedant Nagarkar — ML Engineer Roadmap
Updated April 2026 | Quality over Speed | Complete Edition — All Gaps Filled
Progress Overview
Metric
Status
Notes
Week 0
COMPLETE
7/7 days — all ML/DL algorithms

--- Chunk 2 ---
Tokens: 498
Source: Vedant_Roadmap_FINAL.pdf, Page: 1
WEEK 0 — ML/DL Revision [COMPLETE]
 Day 1: Linear & Logistic Regression
NumPy only | w=2.0 b=1.0 | 100% accuracy | Matches sklearn
exactly
 Day 2: Decision Trees, Random Forest, SVM
4 models compare

--- Chunk 3 ---
Tokens: 126
Source: Vedant_Roadmap_FINAL.pdf, Page: 1
 Day 2: 